# DS605 Lab 1 — Book Data Pipeline
## Scraping, Preprocessing, Visualization, and Reporting



## Task 1 — Data Scraping
### `items.py` — field definitions for each scraped book

In [ ]:

import scrapy


class BookItem(scrapy.Item):
    title = scrapy.Field()
    category = scrapy.Field()
    price = scrapy.Field()
    rating = scrapy.Field()
    availability = scrapy.Field()
    description = scrapy.Field()
    upc = scrapy.Field()
    num_reviews = scrapy.Field()
    product_url = scrapy.Field()

### `books_spider.py` — crawls the catalog and extracts each book's fields

**Note on running this cell:** defining the class below works fine in
Jupyter, but *executing* a live crawl requires Scrapy's own reactor/event
loop, which conflicts with the one Jupyter already runs — so running this
class here will raise a `ReactorNotRestartable` (or similar) error if you
try to launch a `CrawlerProcess` in the same session more than once, and
can behave unpredictably even on the first run. The actual scrape for
this assignment was run from the terminal, inside the Scrapy project
folder:
```bash
scrapy crawl books -o books.json
```
The class is included below as real, complete code (not run from this
notebook) so the full pipeline is documented in one file.

In [ ]:
import scrapy
from book_pipeline.items import BookItem


class BooksSpider(scrapy.Spider):
    name = "books"
    allowed_domains = ["books.toscrape.com"]
    start_urls = ["https://books.toscrape.com/"]

    RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

    MAX_CATALOG_PAGES = 5

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pages_scraped = 0

    def parse(self, response):
        """Parse a catalog/listing page: grab each book's detail link,
        then follow pagination to the next catalog page, up to
        MAX_CATALOG_PAGES total."""

        self.pages_scraped += 1

        for book in response.css("article.product_pod"):
            relative_url = book.css("h3 a::attr(href)").get()
            if relative_url:
                yield response.follow(relative_url, callback=self.parse_book)

        if self.pages_scraped >= self.MAX_CATALOG_PAGES:
            return

        next_page = response.css("li.next a::attr(href)").get()
        if next_page:
            yield response.follow(next_page, callback=self.parse)

    def parse_book(self, response):
        """Parse an individual book detail page and extract all required fields."""

        item = BookItem()

        item["title"] = response.css("div.product_main h1::text").get(default="").strip()
        item["price"] = response.css("p.price_color::text").get(default="").strip()
        item["product_url"] = response.url

        breadcrumb = response.css("ul.breadcrumb li a::text").getall()
        item["category"] = breadcrumb[-1].strip() if len(breadcrumb) >= 3 else None

        rating_class = response.css("p.star-rating::attr(class)").get(default="")
        rating_word = rating_class.replace("star-rating", "").strip()
        item["rating"] = self.RATING_MAP.get(rating_word)

        item["availability"] = " ".join(
            response.css("p.availability::text").getall()
        ).strip()

        item["description"] = response.css(
            "#product_description ~ p::text"
        ).get(default="").strip()

        table_rows = response.css("table.table.table-striped tr")
        table_data = {}
        for row in table_rows:
            key = row.css("th::text").get(default="").strip()
            value = row.css("td::text").get(default="").strip()
            table_data[key] = value

        item["upc"] = table_data.get("UPC")
        item["num_reviews"] = table_data.get("Number of reviews")

        yield item

### Task 1 report — total records, missing values, duplicate UPCs
Reproduces `report.py`, run inline here against the spider's output file.

In [ ]:
import json
import csv
from collections import Counter


def load_records(path):
    if path.endswith(".json"):
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    elif path.endswith(".csv"):
        with open(path, encoding="utf-8", newline="") as f:
            return list(csv.DictReader(f))
    else:
        raise ValueError("File must be .json or .csv")


def books_report(path):
    records = load_records(path)
    total = len(records)
    print(f"Total scraped records: {total}\n")

    fields = ["title", "category", "price", "rating", "availability",
              "description", "upc", "num_reviews", "product_url"]

    print("Missing values per field:")
    for field in fields:
        missing = sum(
            1 for r in records
            if not r.get(field) or str(r.get(field)).strip() == ""
        )
        print(f"  {field:15s}: {missing}")

    print("\nDuplicate UPC check:")
    upcs = [r.get("upc") for r in records if r.get("upc")]
    counts = Counter(upcs)
    duplicates = {upc: c for upc, c in counts.items() if c > 1}
    if duplicates:
        print(f"  Found {len(duplicates)} duplicate UPC value(s):")
        for upc, c in duplicates.items():
            print(f"    {upc}: appears {c} times")
    else:
        print("  No duplicate UPCs found.")


books_report("books.json")

## Task 2 — Data Preprocessing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

sns.set_style("whitegrid")

In [ ]:
with open("books.json", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f"Raw records loaded: {len(df)}")

In [ ]:
text_columns = ["title", "category", "availability", "description"]
for col in text_columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].str.replace(r"\s+", " ", regex=True)

In [ ]:
before = len(df)
df = df.drop_duplicates(subset="upc", keep="first")
print(f"Removed {before - len(df)} duplicate UPC row(s)")

In [ ]:
df["description"] = df["description"].replace("", np.nan)
print(f"Missing descriptions before fill: {df['description'].isna().sum()}")
df["description"] = df["description"].fillna("No description available")

In [ ]:
df["price"] = (
    df["price"]
    .astype(str)
    .str.replace("£", "", regex=False)
    .str.replace("Â", "", regex=False)
    .str.strip()
)
df["price"] = pd.to_numeric(df["price"], errors="coerce")

In [ ]:
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

In [ ]:
df["stock_count"] = (
    df["availability"].str.extract(r"\((\d+)\s*available\)").astype(float)
)
df["stock_count"] = df["stock_count"].fillna(0).astype(int)

### Feature engineering (4 new features)

In [ ]:
df["description_word_count"] = df["description"].str.split().str.len()

In [ ]:
df["price_band"] = pd.cut(
    df["price"],
    bins=[0, 20, 40, df["price"].max() + 1],
    labels=["Low", "Medium", "High"],
)

In [ ]:
df["value_score"] = (df["rating"] / df["price"]).round(3)

In [ ]:
df["recommended"] = (df["rating"] >= 4) & (df["stock_count"] > 0)

In [ ]:
df[["title", "price", "rating", "price_band", "value_score", "recommended"]].head()

In [ ]:
df.to_csv("books_cleaned.csv", index=False)
print("Saved cleaned data to books_cleaned.csv")

## Task 3 — Visualization and Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df["price"], bins=20, kde=True, ax=axes[0, 0], color="steelblue")
axes[0, 0].set_title("Price Distribution")
axes[0, 0].set_xlabel("Price (£)")

sns.countplot(x="rating", data=df, ax=axes[0, 1], palette="viridis")
axes[0, 1].set_title("Rating Distribution")
axes[0, 1].set_xlabel("Rating")

top_categories = df["category"].value_counts().head(10).index
avg_price_cat = (
    df[df["category"].isin(top_categories)]
    .groupby("category")["price"]
    .mean()
    .sort_values()
)
avg_price_cat.plot(kind="barh", ax=axes[1, 0], color="coral")
axes[1, 0].set_title("Average Price by Category (Top 10)")
axes[1, 0].set_xlabel("Average Price (£)")

sns.boxplot(x="rating", y="price", data=df, ax=axes[1, 1], palette="Set2")
axes[1, 1].set_title("Price vs Rating")

plt.tight_layout()
plt.savefig("plots_overview.png", dpi=150)
plt.show()

## Plot summary
#### 1. Price Distribution
###### This plot tells us how the prices of the books are distributed in the dataset. Most of the books lie between a moderate price range of 20 to 40 euros. this tells us that almost all the books are quite affordable. The highest count of books lies somewhere near 20 euros with a count of more than 10.

#### 2. Rating Distribution
##### This plot speaks about how much the books have been rated. the ratings range from 1 star to 5 star. with most of the books luying in the range of either 1 star or 3 star. around 17 books have a rating of 5 star. the ratings are not uniformly distributed.

#### 3. Average Price by Category
##### Here a horizontal bar chart compares the average book price across the top ten categories or genres of the books. the average prices vary for each category. no two or more categories have the same avg. price. But it is observed that the average price for the fiction category is the highest, probably becuase it is one of the top categories bought. 

#### 4. Price VS Rating
##### A box plot was created to compare book prices across different rating levels. We can see that the median prices are almost similar for books belonging to ratings ranging rom 1 to 4,however, 5-star books have a comparatively lower median price, indicating that many highly rated books are reasonably priced.



### Book counts per category
(Backs up the "which categories are most represented" observation in Task 4)

In [ ]:
category_counts = df["category"].value_counts().head(10)

plt.figure(figsize=(10, 6))
category_counts.sort_values().plot(kind="barh", color="mediumseagreen")
plt.title("Book Counts by Category (Top 10)")
plt.xlabel("Number of Books")
plt.ylabel("Category")
plt.tight_layout()
plt.savefig("category_counts.png", dpi=150)
plt.show()

##### this graph talks about how many books does each category contain, with the highest count belonging to sequntial art. with almost 14 books belonging to it; while the lowest count of 4 books belonging to the history,young adult and fantasy genres.

### Word cloud from combined book descriptions
(No customer-review text exists on this site, so descriptions are the
textual source per the assignment instructions.)

In [ ]:
combined_text = " ".join(df["description"].tolist())
wordcloud = WordCloud(
    width=1000, height=500, background_color="white", colormap="viridis"
).generate(combined_text)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud of Book Descriptions")
plt.savefig("wordcloud.png", dpi=150)
plt.show()

##### Word cloud basically depicts how frequently a particular word has appeared in a dataset. Bigger the font size of the word more frequently has it appeared. Words like one,life,time etc must have appeared multile number of times in maybe the titles of the books. Hence they have a very big font size. 

### Summary statistics and insights

In [ ]:
print("=== Summary statistics ===")
print(df[["price", "rating", "stock_count", "value_score"]].describe())

In [ ]:
print("=== Books per category (top 10) ===")
print(df["category"].value_counts().head(10))

In [ ]:
print("=== Highly rated books (rating = 5) ===")
print(df[df["rating"] == 5][["title", "price", "category"]].head(10))

In [ ]:
print("=== Top 10 books by value_score (best value for money) ===")
print(
    df.sort_values("value_score", ascending=False)[
        ["title", "price", "rating", "value_score", "category"]
    ].head(10)
)

In [ ]:
print("=== Average stock count by price band ===")
print(df.groupby("price_band")["stock_count"].mean())

In [ ]:
print("=== Remaining missing values per column ===")
print(df.isna().sum())

##### The findings depict that there is no effect of book price on ratings because all price distributions intersect at all rating levels, with 5-star books having relatively lower median prices. From the category analysis, we can see that there are several categories that make up the bulk of books, while other categories have higher average prices. The value scores of the best-value books based on the value score (Rating ÷ Price) turned out to be high, which means that good books don't have to be expensive. 

#### Limitations
##### One of the major limitations of the scrapped dataset is that we are making a analysis only on a record of 100 books, so it might be difficult to get a concrete or recognizable pattern from the plots or the summary derived. The analysis shows that book price has little influence on ratings, with highly rated books often being reasonably priced and offering good value. Overall, the dataset provides useful insights into pricing, ratings, and category distribution, although the findings are limited by the small, practice-oriented dataset.

